# Cloud Fundamentals

## 1. Core Concepts & Terminology

**Cloud Computing** is the delivery of on-demand computing resources (compute, storage, networking, databases) over the internet with pay-as-you-go pricing.

### Key Principles

- **Consumption-based pricing**: You pay only for what you use (e.g., pay per GB stored, per request, per VM hour).
- **Scalability**: Systems grow (scale out) or shrink (scale in) based on demand automatically.
- **High Availability & Fault Tolerance**: Resources are distributed to avoid single points of failure.
- **Shared Responsibility Model**: Security duties are split between the cloud provider and the customer.
  - Provider manages: physical infrastructure, hypervisors, networking hardware.
  - Customer manages: data, access control, OS patches (varies by service model).

---

## 2. Cloud Service Models

| Model | You Manage | Provider Manages | Example |
|-------|-----------|-----------------|---------|
| **IaaS** | OS, runtime, app, data | Hardware, networking, virtualization | AWS EC2, Azure VMs |
| **PaaS** | App, data | Everything below the app layer | AWS Elastic Beanstalk, Heroku |
| **FaaS** | Function code | Runtime, scaling, infra | AWS Lambda, Azure Functions |
| **SaaS** | Configuration only | Everything | Gmail, Salesforce |

> **Rule of thumb**: The higher you go (IaaS → SaaS), the less you manage but the less control you have.

---

## 3. Cloud Compute Services

### Virtual Machines (VMs)
Full OS instances running on shared hardware. Best for lifting-and-shifting legacy apps.

```
# Example: Launch a Compute Engine VM (gcloud CLI)
gcloud compute instances create my-instance \
  --image=debian-12-bookworm-v20240415 \
  --image-project=debian-cloud \
  --machine-type=e2-micro \
  --metadata=ssh-keys="my-key:$(cat ~/.ssh/id_rsa.pub)"
```

### Containers
Lightweight, portable units that package code + dependencies. Use Docker images, orchestrated by Kubernetes (EKS, GKE, AKS).

- More efficient than VMs (share the host OS kernel).
- Faster startup times, consistent environments.

### Scalability Concepts
- **Vertical scaling**: Increase VM size (more CPU/RAM).
- **Horizontal scaling**: Add more instances behind a load balancer.
- **Auto-scaling**: Cloud triggers scale events based on metrics (CPU > 70% → add instances).

---

## 4. Serverless Computing

Run code **without provisioning or managing servers**. The provider handles execution, scaling, and availability.

### Key Characteristics
- Event-driven: functions trigger on HTTP requests, queue messages, file uploads, etc.
- Stateless: each invocation is independent.
- Billed per execution (duration + memory).

### Example: GCP Cloud Functions (Python)
```python
import functions_framework

@functions_framework.http
def handler(request):
    name = request.get_json(silent=True).get("name", "World") if request.get_json(silent=True) else "World"
    return (f"Hello, {name}!", 200)
```
Deploy and invoke:
```bash
gcloud functions deploy hello \
  --runtime python312 --trigger-http \
  --entry-point handler --source . \
  --allow-unauthenticated

curl -X POST https://REGION-PROJECT_ID.cloudfunctions.net/hello \
  -H "Content-Type: application/json" \
  -d '{"name":"Ana"}'
```

### Common Serverless Use Cases
- REST APIs (Lambda + API Gateway)
- Data processing pipelines (S3 event → Lambda → DynamoDB)
- Scheduled tasks (CloudWatch Events / EventBridge)

---

## 5. Infrastructure as Code (IaC)

Define and provision infrastructure using code instead of manual clicks. Enables version control, reproducibility, and automation.

| Tool | Provider | Language |
|------|----------|----------|
| CloudFormation | AWS | YAML/JSON |
| Terraform | Multi-cloud | HCL |
| Bicep | Azure | DSL |
| Pulumi | Multi-cloud | Python/TS/Go |

**Example (Terraform — create a GCS bucket):**
```hcl
resource "google_storage_bucket" "my_bucket" {
  name     = "my-app-assets-2025"
  location = "US"
}
```
---

## 6. Cloud Storage Services

| Type | Description | Use Case | GCP Example |
|------|-------------|----------|-------------|
| **Object Storage** | Flat storage for files/blobs | Images, backups, static assets | Cloud Storage |
| **Block Storage** | Low-latency disk volumes | VM disks, databases | Persistent Disk |
| **File Storage** | Shared NFS/SMB filesystem | Shared app configs, media | Filestore |

**Key object storage concepts:**
- Globally unique bucket names.
- Objects stored with a key (path-like string).
- Versioning, lifecycle policies, and access control via bucket policies.

---

## 7. Cloud Database Services

### Relational (SQL)
Structured data with ACID transactions. Managed services handle backups, patching, and replication.

- AWS RDS (MySQL, PostgreSQL, SQL Server)
- Azure SQL Database
- Google Cloud SQL

```sql
-- Example: create a users table on RDS PostgreSQL
CREATE TABLE users (
  id SERIAL PRIMARY KEY,
  email TEXT UNIQUE NOT NULL,
  created_at TIMESTAMPTZ DEFAULT NOW()
);
```

### NoSQL
Flexible schemas, optimized for scale and specific access patterns.

| Type | GCP Service | Best For |
|------|-------------|----------|
| Key-Value / Document | Firestore | Shopping carts, user sessions |
| Wide-Column | Bigtable | Time-series, IoT |
| Graph | Spanner (Graph) | Social networks, recommendations |

**Firestore Example:**
```python
from google.cloud import firestore
db = firestore.Client()
db.collection("Users").document("u1").set({"userId": "u1", "email": "ana@example.com"})
```

### Key Concepts
- **Replication**: Multi-AZ for high availability; read replicas for scaling reads.
- **Backup**: Automated snapshots + point-in-time recovery.
- **Consistency**: SQL = strong consistency; NoSQL often offers eventual consistency.

---

## 8. Cloud Integration & Messaging Services

Enable loosely coupled, asynchronous communication between services.

| Pattern | Service | Use Case |
|---------|---------|----------|
| **Queue (P2P)** | Cloud Tasks | Decouple producers from consumers |
| **Pub/Sub** | Cloud Pub/Sub | Fan-out notifications to many subscribers |
| **Event Bus** | Eventarc | Event-driven routing between services |
| **Streaming** | Dataflow / Pub/Sub | Real-time data pipelines |

**Example flow — order processing:**
```
Order Service → Cloud Tasks Queue → Fulfillment Cloud Function
                                  → Notification Cloud Function
```

---

## 9. Cloud Security Best Practices

### Identity & Access Management (IAM)
- Apply **least privilege**: grant only the permissions needed.
- Use roles (not root/admin accounts) for services and users.
- Enable **MFA** on all human accounts.

### Data Protection
- Encrypt data **at rest** (AES-256, Cloud KMS keys) and **in transit** (TLS 1.2+).
- Never store secrets in source code — use **Secret Manager** or **Cloud Storage** with restricted access.

### Network Security
- Place resources in **private subnets**; expose only what's needed via load balancers.
- Use **VPC Firewall Rules** and **Hierarchical Firewall Policies** as layered firewalls.
- Enable **VPC Flow Logs** for auditing.

### Monitoring & Compliance
- Enable **Cloud Audit Logs** (audit log of API calls) and **Cloud Monitoring** (metrics/alarms).
- Use **Security Command Center** or **Organization Policy Service** to enforce compliance rules automatically.
- Rotate credentials and review IAM permissions regularly.

---

## 10. Serverless vs. Serverfull — Quick Comparison

| | Serverless (FaaS) | Serverfull (VMs/Containers) |
|--|-------------------|-----------------------------|
| **Management** | None (provider handles all) | You manage OS, scaling, patches |
| **Scaling** | Automatic, per request | Configured auto-scaling groups |
| **Cost model** | Pay per invocation | Pay per running instance |
| **Cold starts** | Yes (~ms to seconds) | No |
| **Best for** | Event-driven, variable load | Long-running, stateful workloads |

---

## 11. Multi-Cloud Considerations

Using 2+ cloud providers reduces vendor lock-in and improves resilience, but adds operational complexity.

- **AWS**: Broadest service catalog; most mature ecosystem.
- **Azure**: Strong enterprise integration (Active Directory, Office 365).
- **GCP**: Leading in data/ML (BigQuery, Vertex AI, Kubernetes origin).

Common strategy: primary workloads on one provider, DR or specialized services on another.

---

> **Summary**: Cloud fundamentals revolve around understanding the right service model (IaaS/PaaS/FaaS/SaaS) for each workload, leveraging managed services for storage and databases, adopting event-driven patterns for integration, and applying security-by-default at every layer.